In [0]:
# Verificando os arquivos disponíveis no volume
volume_path = "/Volumes/workspace/default/olist_raw"

dbutils.fs.ls(volume_path)

In [0]:
# =============================================================================
# CAMADA BRONZE - Ingestão dos dados brutos
# =============================================================================
# Objetivo: Ler os arquivos CSV originais do volume e salvá-los como tabelas
# Delta no Databricks, sem nenhuma transformação. A camada Bronze preserva
# os dados exatamente como vieram da fonte, adicionando apenas metadados
# de controle (arquivo de origem e data de ingestão).
# =============================================================================

from pyspark.sql import functions as F
from datetime import date

# Caminho do volume onde os CSVs foram carregados manualmente
volume_path = "/Volumes/workspace/default/olist_raw"

# Data de hoje usada como metadado de controle em todas as tabelas
ingestion_date = str(date.today())

# Dicionário mapeando cada arquivo CSV ao nome da tabela Bronze correspondente
# Chave: nome do arquivo | Valor: nome da tabela Delta de destino
arquivos = {
    "olist_customers_dataset.csv":           "bronze_customers",
    "olist_geolocation_dataset.csv":         "bronze_geolocation",
    "olist_order_items_dataset.csv":         "bronze_order_items",
    "olist_order_payments_dataset.csv":      "bronze_order_payments",
    "olist_order_reviews_dataset.csv":       "bronze_order_reviews",
    "olist_orders_dataset.csv":              "bronze_orders",
    "olist_products_dataset.csv":            "bronze_products",
    "olist_sellers_dataset.csv":             "bronze_sellers",
    "product_category_name_translation.csv": "bronze_category_translation",
}

# Loop principal: para cada arquivo, lê o CSV e salva como tabela Delta
for arquivo, tabela in arquivos.items():

    # Leitura do CSV com inferência automática de schema e cabeçalho.
    #
    # multiLine=true: necessário porque alguns arquivos (notadamente o de
    # reviews) contêm quebras de linha dentro de campos de texto livre, como
    # os comentários escritos pelos clientes. Sem essa opção o Spark encerra
    # o registro na quebra de linha e desalinha todas as colunas seguintes,
    # fazendo com que trechos de texto acabem em colunas de data ou nota.
    #
    # quote e escape: garantem o tratamento correto de aspas e vírgulas
    # dentro dos campos de texto livre.
    df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .option("multiLine", "true")
              .option("quote", '"')
              .option("escape", '"')
              .csv(f"{volume_path}/{arquivo}"))

    # Adição de colunas de metadados de controle
    # _source_file: rastreabilidade, saber de qual arquivo veio cada registro
    # _ingestion_date: quando esse dado entrou no pipeline
    df = (df.withColumn("_source_file", F.lit(arquivo))
            .withColumn("_ingestion_date", F.lit(ingestion_date)))

    # Gravação como tabela Delta no schema default do catalog workspace
    # mode("overwrite"): substitui a tabela se já existir (seguro para reexecução)
    # overwriteSchema=true: permite substituir também o schema da tabela, e não
    # apenas os dados. Necessário porque a correção na leitura do CSV alterou
    # os tipos inferidos em algumas colunas em relação à execução anterior.
    # format("delta"): formato padrão do Databricks, suporta ACID e time travel
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.default.{tabela}")

    # Confirmação de sucesso com contagem de registros
    print(f"✓ {tabela} salva com {df.count()} registros")